In [96]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt


In [97]:
# Read the laod data
data = pd.read_csv('/content/train.csv')

In [98]:
data.head()

,label,pixel0,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,...,pixel774,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783
0,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,4,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [99]:
data = np.array(data)
# After this, 'data' is a NumPy array. Subsequent steps in the notebook
# prepare 'X_train' from this data. The dimension '784' in 'X_train[:,0].shape'
# signifies the number of features (pixels) for a single image, after the label
# column has been separated from the original 785 columns.

In [100]:
print(data)


[[1 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [1 0 0 ... 0 0 0]
 ...
 [7 0 0 ... 0 0 0]
 [6 0 0 ... 0 0 0]
 [9 0 0 ... 0 0 0]]


In [101]:
# Suffle is the method for data preparation
m,n = data.shape
np.random.shuffle(data)

data_dev = data[0:1000].T
Y_dev = data_dev[0]
X_dev = data_dev[1:n]

data_train = data[1000:m].T
Y_train = data_train[0]
X_train = data_train[1:n]


In [102]:
X_train[:,0].shape

(784,)

In [103]:
def init_params():
  w1 = np.random.rand(10,784) - 0.5 # 0.5 is bias
  b1 = np.random.rand(10,1) - 0.5
  w2 = np.random.rand(10,10) - 0.5
  b2 = np.random.rand(10,1) - 0.5
  return w1,b1,w2,b2
# the valude of relu is within 0-1
def relu(z):
  return np.maximum(z,0)

def softmax(z):
  # Subtract the maximum value for numerical stability
  exp_z = np.exp(z - np.max(z, axis=0, keepdims=True))
  return exp_z / np.sum(exp_z, axis=0, keepdims=True)

# model trying to go forward to find the output
def forward_prop(w1,b1,w2,b2,X):
  z1 = w1.dot(X) + b1
  a1 = relu(z1)
  z2 = w2.dot(a1) + b2
  a2 = softmax(z2)
  return z1,a1,z2,a2

def one_hot_Y(Y):
  one_hot_Y = np.zeros((Y.size,Y.max()+1))
  one_hot_Y[np.arange(Y.size),Y] = 1
  one_hot_Y = one_hot_Y.T
  return one_hot_Y

def deriv_relu(z):
  return z > 0
# model is trying to mapping back whether the output is correct or not
def back_prop(z1,a1,z2,a2,w1,w2,X,Y):
  m = Y.size
  Y_one_hot = one_hot_Y(Y)
  #d is refering to derivative
  dz2 = a2 - Y_one_hot
  dw2 = 1/m * dz2.dot(a1.T)
  # Sum along axis=1 (samples) to get gradients for biases, keepdims for broadcasting
  db2 = 1/m * np.sum(dz2, axis=1, keepdims=True)
  dz1 = w2.T.dot(dz2) * deriv_relu(z1)
  dw1 = 1/m * dz1.dot(X.T)
  # Sum along axis=1 (samples) to get gradients for biases, keepdims for broadcasting
  db1 = 1/m * np.sum(dz1, axis=1, keepdims=True)
  return dw1,db1,dw2,db2
#while the model back propagation , it also trying to update a numpy for debug

def update_params(w1,b1,w2,b2,dw1,db1,dw2,db2,alpha):
  w1 = w1 - alpha * dw1
  b1 = b1 - alpha * db1
  w2 = w2 - alpha * dw2
  b2 = b2 - alpha * db2
  return w1,b1,w2,b2

In [104]:
def get_predictions(a2):
  return np.argmax(a2,0)

def get_accuracy(predictions,Y):
  print(predictions,Y)
  return np.sum(predictions == Y) / Y.size


def gradue_descent(X,Y,alpha,iterations):

  w1,b1,w2,b2 = init_params()
  for i in range(iterations):
    z1,a1,z2,a2 = forward_prop(w1,b1,w2,b2,X)
    dw1,db1,dw2,db2 = back_prop(z1,a1,z2,a2,w1,w2,X,Y)
    w1,b1,w2,b2 = update_params(w1,b1,w2,b2,dw1,db1,dw2,db2,alpha)
    if i % 10 == 0: # meaning print the number of iteration in every 50 times
      print("Iteration: ", i)
      predictions = get_predictions(a2)
      print(get_accuracy(predictions,Y))
  return w1,b1,w2,b2

In [105]:
w1,b1,w2,b2 = gradue_descent(X_train,Y_train,0.1,100)

Iteration:  0
[4 6 4 ... 8 2 4] [5 2 6 ... 3 8 1]
0.08717073170731707
Iteration:  10
[6 6 6 ... 6 6 6] [5 2 6 ... 3 8 1]
0.09821951219512196
Iteration:  20
[6 6 6 ... 6 6 6] [5 2 6 ... 3 8 1]
0.09821951219512196
Iteration:  30
[6 6 6 ... 6 6 6] [5 2 6 ... 3 8 1]
0.09821951219512196
Iteration:  40
[6 6 6 ... 6 6 6] [5 2 6 ... 3 8 1]
0.09821951219512196
Iteration:  50
[6 6 6 ... 6 6 6] [5 2 6 ... 3 8 1]
0.09821951219512196
Iteration:  60
[6 6 6 ... 6 6 6] [5 2 6 ... 3 8 1]
0.09821951219512196
Iteration:  70
[6 6 6 ... 6 6 6] [5 2 6 ... 3 8 1]
0.09821951219512196
Iteration:  80
[6 6 6 ... 6 6 6] [5 2 6 ... 3 8 1]
0.09821951219512196
Iteration:  90
[6 6 6 ... 6 6 6] [5 2 6 ... 3 8 1]
0.09821951219512196


In [106]:
m,n

(42000, 785)

In [107]:
a = [1,2,3,4,5]

In [108]:
a[:0]

[]